# Part 4 — Per-patient causal inference (CORNETO/CARNIVAL)

For each cohort patient we restrict the signed PKN to the kinase → kinase∪TF
subspace, then solve a signed-flow CARNIVAL problem: the strongest kinase
activities are inputs, the strongest ±TF activities are outputs. The sparsity
weight `lambd` is swept over a small grid. Solver failures and empty solutions
are recorded, not hidden. Inferred subnetworks are model-based hypotheses.

In [ ]:
import time

import numpy as np
import pandas as pd

import corneto as cn
from corneto.methods.signalling.carnival import multi_carnival

from uc1_common import *  # noqa: F403

tf_es = pd.read_parquet(TF_ES)
kin_es = pd.read_parquet(KIN_ES)
cohort_definitions = load_cohorts()
G = load()
analysis_patients = sorted(set().union(
    *(set(cohort_definitions[label]) for label in SENSITIVITY_COHORT_LABELS if label in cohort_definitions)
))
assert analysis_patients, 'No patients were selected for the analysis cohorts.'

## Signed PKN restricted to the kinase → kinase∪TF subnet

A modelling choice for tractability — not a claim that other signaling nodes are
irrelevant.

In [ ]:
kin_set = set(map(str, kin_es.columns))
tf_set = set(map(str, tf_es.columns))

pkn_filt = pkn_edges(G)
pkn_filt = pkn_filt[pkn_filt['source'].isin(kin_set) & pkn_filt['target'].isin(kin_set | tf_set)].copy()
pkn_filt['sign'] = np.where(
    pkn_filt['consensus_stimulation'].fillna(False).astype(bool), 1,
    np.where(pkn_filt['consensus_inhibition'].fillna(False).astype(bool), -1, 0),
)
pkn_filt = pkn_filt[pkn_filt['sign'] != 0].copy()

C_filt = cn.Graph.from_tuples(pkn_filt[['source', 'sign', 'target']].apply(tuple, axis=1).tolist())
filtered_pkn_summary = pd.DataFrame({'metric': ['filtered_edges', 'filtered_nodes'],
                                     'value': [len(pkn_filt), C_filt.num_vertices]})
filtered_pkn_summary.to_csv(TABLES / 'filtered_pkn_summary.csv', index=False)
filtered_pkn_summary

## Solve per patient across the regularization grid

`build_patient_io` picks the top kinase inputs and ±TF outputs;
`multi_carnival` solves the signed-flow problem, with a solver fallback list; and
`extract_active_network` reads the active nodes/edges back out above the signal
threshold. Every patient × lambda outcome is recorded.

In [ ]:
def build_patient_io(patient_id):
    if patient_id not in kin_es.index or patient_id not in tf_es.index:
        return None
    kr, tr = kin_es.loc[patient_id], tf_es.loc[patient_id]
    kin_top = kr[kr.abs() >= KIN_TH].abs().nlargest(TOP_K_KIN).index
    tf_out = [*tr[tr >= TF_TH].nlargest(TOP_TF_POS).index, *tr[tr <= -TF_TH].nsmallest(TOP_TF_NEG).index]
    inputs = {k: float(np.sign(kr[k])) for k in kin_top}
    outputs = {tf: float(tr[tf]) for tf in tf_out}
    return {'input': inputs, 'output': outputs} if inputs and outputs else None


def solve_with_fallback(graph, payload, lambd):
    P, G_p, meta = multi_carnival(graph, payload, lambd=lambd)
    last_error = None
    for solver in SOLVER_CANDIDATES:
        try:
            return P, G_p, meta, P.solve(solver=solver), solver, None
        except Exception as exc:
            last_error = f'{type(exc).__name__}: {exc}'
    return P, G_p, meta, None, None, last_error


def extract_active_network(P, G_p, threshold):
    vvals = np.asarray(P.expr.vertex_value.value)
    evals = np.asarray(P.expr.edge_value.value)
    if vvals.ndim == 2:
        vvals = vvals[:, 0]
    if evals.ndim == 2:
        evals = evals[:, 0]

    node_signal = {}
    for idx, node in enumerate(G_p.V):
        node = str(node)
        if not node.startswith('_') and G.has_vertex(node) and abs(float(vvals[idx])) >= threshold:
            node_signal[node] = float(vvals[idx])

    edge_signal = {}
    for idx, (src_fs, tgt_fs) in enumerate(G_p.E):
        if abs(float(evals[idx])) < threshold:
            continue
        src = [str(x) for x in src_fs if not str(x).startswith('_')]
        tgt = [str(x) for x in tgt_fs if not str(x).startswith('_')]
        if src and tgt and G.has_vertex(src[0]) and G.has_vertex(tgt[0]):
            edge_signal[(src[0], tgt[0])] = float(evals[idx])
    return node_signal, edge_signal


patient_io_map = {p: io for p in analysis_patients if (io := build_patient_io(p))}
assert patient_io_map, 'No patients had valid kinase-input and TF-output constraints for CARNIVAL.'

carnival_rows, active_networks_by_lambda = [], {}
for lambd in REGULARIZATION_GRID:
    active_networks_by_lambda[lambd] = {}
    for patient_id in sorted(patient_io_map):
        io = patient_io_map[patient_id]
        t0 = time.time()
        P, G_p, meta, sol, solver, error = solve_with_fallback(C_filt, {patient_id: io}, lambd)
        runtime = time.time() - t0

        if sol is None:
            rec = {'solver_status': 'solve_failed', 'solver_ok': False, 'objective_value': np.nan,
                   'node_signal': {}, 'edge_signal': {}, 'error': error}
        else:
            status = str(getattr(sol, 'status', 'unknown')).strip().lower()
            node_signal, edge_signal = extract_active_network(P, G_p, CORNETO_SIG_THRESHOLD)
            rec = {'solver_status': status, 'solver_ok': status in ACCEPTABLE_SOLVER_STATUSES,
                   'objective_value': float(getattr(sol, 'value', np.nan)) if getattr(sol, 'value', None) is not None else np.nan,
                   'node_signal': node_signal, 'edge_signal': edge_signal, 'error': None}
        rec.update({'patient_id': patient_id, 'lambd': lambd, 'solver': solver, 'runtime_s': runtime,
                    'active_node_count': len(rec['node_signal']), 'active_edge_count': len(rec['edge_signal']),
                    'n_inputs': len(io['input']), 'n_outputs': len(io['output'])})
        active_networks_by_lambda[lambd][patient_id] = rec
        carnival_rows.append({k: rec[k] for k in
            ['patient_id', 'lambd', 'solver', 'solver_status', 'solver_ok', 'objective_value',
             'runtime_s', 'active_node_count', 'active_edge_count', 'n_inputs', 'n_outputs', 'error']})

carnival_summary = pd.DataFrame(carnival_rows)
carnival_summary.to_csv(TABLES / 'carnival_patient_summary.csv', index=False)
lambda_summary = (carnival_summary.groupby('lambd', dropna=False).agg(
    n_patients=('patient_id', 'nunique'), solver_ok_fraction=('solver_ok', 'mean'),
    mean_active_nodes=('active_node_count', 'mean'), mean_active_edges=('active_edge_count', 'mean'),
    median_runtime_s=('runtime_s', 'median')).reset_index())
lambda_summary.to_csv(TABLES / 'carnival_lambda_summary.csv', index=False)

assert PRIMARY_LAMBDA in active_networks_by_lambda
assert carnival_summary['solver_ok'].any(), 'No successful/feasible CARNIVAL solution was found.'
save_carnival(active_networks_by_lambda)
lambda_summary

## Write per-patient subnetworks back into the graph

Only the primary-regularization solutions for solved, non-empty patients are
materialized as intra-layer edges. Failures stay in the summary table.

In [ ]:
primary_patient_networks = active_networks_by_lambda[PRIMARY_LAMBDA]

written_rows = []
for patient_id, result in primary_patient_networks.items():
    if not result['solver_ok']:
        written_rows.append({'patient_id': patient_id, 'written': False, 'reason': result['solver_status']})
        continue
    if result['active_node_count'] == 0:
        written_rows.append({'patient_id': patient_id, 'written': False, 'reason': 'empty_solution'})
        continue

    aa = (patient_id,)
    endpoints = set(result['node_signal'])
    for src, tgt in result['edge_signal']:
        endpoints.update((src, tgt))
    G.add_vertices(sorted(endpoints), layer=aa)
    for node_id in sorted(endpoints):
        G.layers.set_vertex_layer_attrs(node_id, aa, corneto_signal=float(result['node_signal'].get(node_id, 0.0)))

    specs = [{'source': (src, aa), 'target': (tgt, aa), 'weight': float(w)}
             for (src, tgt), w in sorted(result['edge_signal'].items())]
    if specs:
        G.add_edges(specs)
    written_rows.append({'patient_id': patient_id, 'written': True, 'reason': 'ok',
                         'active_nodes': result['active_node_count'], 'active_edges': result['active_edge_count']})

written_summary = pd.DataFrame(written_rows)
written_summary.to_csv(TABLES / 'written_patient_networks.csv', index=False)
G.history.snapshot('after_patient_specific_causal_networks')
written_summary.head()

## Prior vs inferred, as two slices on one graph

Every CARNIVAL edge is registered in a `carnival_inferred` slice alongside the
default `omnipath_pkn`. Toggling the active slice switches between the prior and
the inferred topology with no rebuild.

In [ ]:
CARNIVAL_SLICE = 'carnival_inferred'
if not G.slices.exists(CARNIVAL_SLICE):
    G.slices.add(CARNIVAL_SLICE)

# Map each (src_supra, tgt_supra) back to its edge id via the multilayer index.
src_to_edges = G._src_to_edges
carnival_eids = []
for patient_id, result in primary_patient_networks.items():
    if not result['solver_ok']:
        continue
    aa = (patient_id,)
    for src, tgt in result['edge_signal']:
        tkey = (tgt, aa)
        for eid in src_to_edges.get((src, aa), ()):
            rec = G._edges.get(eid)
            if rec is not None and rec.tgt == tkey:
                carnival_eids.append(eid)
                break

G.slices.add_edges(CARNIVAL_SLICE, carnival_eids)
G.history.snapshot('after_carnival_slice')
G.write(str(SNAPSHOT), overwrite=True)
print(f'omnipath_pkn      : {len(G.slices.edges("omnipath_pkn")):>7,} edges')
print(f'carnival_inferred : {len(G.slices.edges(CARNIVAL_SLICE)):>7,} edges')